# Faiyad_PartB_Q2_LogisticRegression
## CT107-3-3-TXSA | Group Assignment Part B 


## Q2 — Logistic Regression — Baseline Sentiment Classification Model

## Install and Import Required Libraries

In [ ]:
# Install required libraries
# scikit-learn — model, vectoriser and evaluation metrics
# pandas       — loading CSV files
# matplotlib   — styled table and chart visualisations
# seaborn      — confusion matrix heatmaps
# numpy        — bar chart positioning
%pip install scikit-learn pandas matplotlib seaborn numpy

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings('ignore')

# Global plot style
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'font.size'         : 12,
    'axes.titlesize'    : 14,
    'axes.titleweight'  : 'bold',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'figure.dpi'        : 150,
})

# Colour palette — Green=Positive | Red=Negative | Orange=Neutral
COLORS  = {'positive': '#4CAF50', 'negative': '#F44336', 'neutral': '#FF9800'}
PALETTE = ['#4CAF50', '#F44336', '#FF9800']
LABELS  = ['positive', 'negative', 'neutral']

print('Libraries imported successfully.')
print(f'Sentiment classes: {LABELS}')

## Load the Preprocessed Train and Test Sets

In [ ]:
# Load the cleaned train and test CSV files 
train_df = pd.read_csv('../Data/train_set.csv')
test_df  = pd.read_csv('../Data/test_set.csv')

# Separate features (X) from labels (y)
# fillna('') handles any residual empty strings in the 'Summary_clean' column, ensuring the vectorizer receives valid input
X_train = train_df['Summary_clean'].fillna('')
y_train = train_df['Sentiment']
X_test  = test_df['Summary_clean'].fillna('')
y_test  = test_df['Sentiment']

# Confirm sizes and class distributions match the EDA output
print(f'Training set : {len(X_train):,} rows')
print(f'Test set     : {len(X_test):,} rows')
print(f'\nClass distribution — Training set:')
print(y_train.value_counts())
print(f'\nClass distribution — Test set:')
print(y_test.value_counts())

## TF-IDF Vectorisation

In [ ]:
# TF-IDF converts raw text into weighted numerical features
# TF  — how often a word appears in a document
# IDF — down-weights words that appear across many documents
# This addresses the vocabulary overlap identified in EDA where words like
# 'good' and 'quality' appeared in both positive and negative reviews

# Default parameters used at baseline
tfidf = TfidfVectorizer()

# fit_transform on training data only — learns vocabulary and IDF weights
# Fitting only on training data prevents data leakage into the test set
X_train_tfidf = tfidf.fit_transform(X_train)

# transform on test data — applies training vocabulary without refitting
# Out-of-vocabulary words from the test set are ignored
X_test_tfidf  = tfidf.transform(X_test)

print(f'Vocabulary size       : {len(tfidf.vocabulary_):,} unique terms')
print(f'Training matrix shape : {X_train_tfidf.shape}')
print(f'Test matrix shape     : {X_test_tfidf.shape}')
print(f'Matrix sparsity       : {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.4%}')

## Build the Logistic Regression Baseline Model

In [ ]:
# Logistic Regression estimates class probabilities using a linear decision boundary
# For three-class classification sklearn uses One-vs-Rest (OvR) by default —
# one binary classifier is trained per class and the highest probability wins


# All remaining hyperparameters (C, solver, penalty, max_iter) 
lr_model = LogisticRegression(
    class_weight='balanced',
    random_state=42
)

# Train the model on the TF-IDF transformed training set
lr_model.fit(X_train_tfidf, y_train)

# Generate predictions on the unseen test set
y_pred = lr_model.predict(X_test_tfidf)

print('Logistic Regression baseline model trained successfully.')
print(f'\nFull model parameters:')
for param, value in lr_model.get_params().items():
    print(f'  {param:20s}: {value}')

## Classification Report

In [ ]:
# Raw sklearn classification report — per-class precision, recall, F1 and support
print('=== LOGISTIC REGRESSION — BASELINE CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, labels=LABELS, target_names=LABELS))

In [ ]:
# Extract report as dictionary for use in styled table below
report = classification_report(y_test, y_pred, target_names=LABELS, output_dict=True)

# Compute overall metrics
# Macro averaging treats all classes equally — most appropriate for imbalanced data
# as it prevents the dominant positive class from inflating the overall score
accuracy    = accuracy_score(y_test, y_pred)
macro_p     = precision_score(y_test, y_pred, average='macro')
macro_r     = recall_score(y_test, y_pred, average='macro')
macro_f1    = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

print('=== OVERALL METRICS ===')
print(f'Accuracy          : {accuracy:.4f}')
print(f'Macro Precision   : {macro_p:.4f}')
print(f'Macro Recall      : {macro_r:.4f}')
print(f'Macro F1-Score    : {macro_f1:.4f}')
print(f'Weighted F1-Score : {weighted_f1:.4f}')

In [ ]:
# Styled classification report table 
# Rows 1-3: per-class results | Rows 4-6: averages and accuracy
table_data = []
for label in LABELS:
    r = report[label]
    table_data.append([
        label.capitalize(),
        f'{r["precision"]:.4f}',
        f'{r["recall"]:.4f}',
        f'{r["f1-score"]:.4f}',
        f'{int(r["support"]):,}'
    ])

table_data.append(['Macro Avg',
    f'{report["macro avg"]["precision"]:.4f}',
    f'{report["macro avg"]["recall"]:.4f}',
    f'{report["macro avg"]["f1-score"]:.4f}',
    f'{int(report["macro avg"]["support"]):,}'])

table_data.append(['Weighted Avg',
    f'{report["weighted avg"]["precision"]:.4f}',
    f'{report["weighted avg"]["recall"]:.4f}',
    f'{report["weighted avg"]["f1-score"]:.4f}',
    f'{int(report["weighted avg"]["support"]):,}'])

table_data.append(['Accuracy', '', '', f'{accuracy:.4f}', f'{len(y_test):,}'])

fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')
t = ax.table(
    cellText=table_data,
    colLabels=['Class', 'Precision', 'Recall', 'F1-Score', 'Support'],
    cellLoc='center', loc='center',
    colWidths=[0.22, 0.18, 0.18, 0.18, 0.18]
)
t.auto_set_font_size(False)
t.set_fontsize(11)
t.scale(1, 2.0)

# Header = blue | Class rows = sentiment colour in first column
# Average rows = grey to distinguish from class rows
row_class_colors = ['#4CAF50', '#F44336', '#FF9800']
for (r, c), cell in t.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', fontweight='bold')
    elif r <= 3 and c == 0:
        cell.set_facecolor(row_class_colors[r - 1])
        cell.set_text_props(color='white', fontweight='bold')
    elif r > 3:
        cell.set_facecolor('#CFD8DC')
        cell.set_text_props(fontweight='bold')
    else:
        cell.set_facecolor('#E3F2FD' if r % 2 == 0 else '#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title('Logistic Regression — Baseline Classification Report',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## Confusion Matrix

In [ ]:
# Compute confusion matrix — rows = true labels | columns = predicted labels
# Diagonal cells = correct predictions | off-diagonal = misclassifications
cm = confusion_matrix(y_test, y_pred, labels=LABELS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — raw counts: shows absolute number of predictions per cell
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[l.capitalize() for l in LABELS],
    yticklabels=[l.capitalize() for l in LABELS],
    linewidths=0.5, linecolor='white',
    ax=axes[0], annot_kws={'size': 12, 'weight': 'bold'}
)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontweight='bold')
axes[0].set_ylabel('True Label', fontweight='bold')

# Right — row-normalised: divides each row by total true instances in that class
# More meaningful than raw counts when class sizes differ significantly
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Greens',
    xticklabels=[l.capitalize() for l in LABELS],
    yticklabels=[l.capitalize() for l in LABELS],
    linewidths=0.5, linecolor='white',
    ax=axes[1], annot_kws={'size': 12, 'weight': 'bold'}
)
axes[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontweight='bold')
axes[1].set_ylabel('True Label', fontweight='bold')

plt.suptitle('Logistic Regression — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Per-Class Metric Visualisation

In [ ]:
# Grouped bar chart comparing Precision, Recall and F1-Score per sentiment class
# Identifies which classes the model struggles with and whether the issue
# is in precision (false positives) or recall (missed true instances)
metrics_data = {
    'Precision': [report[l]['precision'] for l in LABELS],
    'Recall'   : [report[l]['recall']    for l in LABELS],
    'F1-Score' : [report[l]['f1-score']  for l in LABELS]
}

x     = np.arange(len(LABELS))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width, metrics_data['Precision'], width,
               label='Precision', color='#1565C0', edgecolor='white')
bars2 = ax.bar(x,          metrics_data['Recall'],    width,
               label='Recall',    color='#42A5F5', edgecolor='white')
bars3 = ax.bar(x + width,  metrics_data['F1-Score'],  width,
               label='F1-Score',  color='#90CAF9', edgecolor='white')

# Annotate each bar with its exact score above the bar
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.2f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold'
        )

ax.set_xticks(x)
ax.set_xticklabels([l.capitalize() for l in LABELS], fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Logistic Regression — Per-Class Precision, Recall and F1-Score',
             fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 2.8 Baseline Results Summary

In [ ]:
# Summary table of all key baseline results and settings
# Serves as the reference point before hyperparameter tuning in Q3
summary_data = [
    ['Model',             'Logistic Regression (Baseline)'],
    ['Accuracy',          f'{accuracy:.4f}'],
    ['Macro Precision',   f'{macro_p:.4f}'],
    ['Macro Recall',      f'{macro_r:.4f}'],
    ['Macro F1-Score',    f'{macro_f1:.4f}'],
    ['Weighted F1-Score', f'{weighted_f1:.4f}'],
    ['class_weight',      'balanced'],
    ['random_state',      '42'],
    ['TF-IDF Settings',   'Default'],
    ['LR Settings',       'Default'],
]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axis('off')
t = ax.table(
    cellText=summary_data,
    colLabels=['Property', 'Value'],
    cellLoc='left', loc='center',
    colWidths=[0.45, 0.55]
)
t.auto_set_font_size(False)
t.set_fontsize(11)
t.scale(1, 1.9)

for (r, c), cell in t.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    else:
        cell.set_facecolor('#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title('Logistic Regression — Baseline Results Summary',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print('=' * 55)
print(' BASELINE EVALUATION COMPLETE')
print('=' * 55)
print(f'Macro F1-Score (baseline) : {macro_f1:.4f}')
print(f'Accuracy       (baseline) : {accuracy:.4f}')
print('=' * 55)

---


## Q3 — Hyperparameter Tuning 

### Q3.0 — Additional Imports

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.feature_selection import SelectKBest, chi2
from collections import Counter
import time
import warnings
warnings.filterwarnings('ignore')

# Colour constants (consistent with Q2 palette)
BG   = '#F8F9FA'
DARK = '#1A237E'
BLUE = '#1565C0'

print(f'Training set : {len(X_train):,} rows')
print(f'Test set     : {len(X_test):,} rows')
print(f'Classes      : {y_train.value_counts().to_dict()}')

---
### Part 1 — Hyperparameter EDA

In [ ]:
tfidf_base   = TfidfVectorizer()
X_base       = tfidf_base.fit_transform(X_train)
vocab_base   = tfidf_base.get_feature_names_out()
doc_freq_base = np.diff(X_base.tocsc().indptr)        # per-term doc count
total_vocab   = len(doc_freq_base)
total_docs    = X_base.shape[0]

print(f'Base vocabulary  : {total_vocab:,} unique terms')
print(f'Training docs    : {total_docs:,}')

#### Class Imbalance → `lr__C`

In [ ]:
vc     = y_train.value_counts()
labels_plot = ['Positive', 'Negative', 'Neutral']
counts = [vc['positive'], vc['negative'], vc['neutral']]
pcts   = [c / len(y_train) * 100 for c in counts]
colors = ['#4CAF50', '#F44336', '#FF9800']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('white')

bars = axes[0].bar(labels_plot, counts, color=colors,
                   width=0.55, edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Class Distribution (Training Set)', fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
for bar, cnt, pct in zip(bars, counts, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 300,
                 f'{cnt:,}\n({pct:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, max(counts) * 1.22)

wedges, texts, autotexts = axes[1].pie(
    counts, labels=labels_plot, colors=colors, autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 11})
for at in autotexts:
    at.set_fontweight('bold')
axes[1].set_title('Class Proportion — Training Set', fontweight='bold')

fig.text(0.5, 0.01,
    f'Imbalance ratio {pcts[0]/pcts[2]:.1f} : {pcts[1]/pcts[2]:.1f} : 1  |  '
    f'Majority (positive) = {pcts[0]:.1f}%  |  Minority (neutral) = {pcts[2]:.1f}%\n'
    'Low C applies stronger regularisation — prevents over-commitment to majority class\n'
    'Justifies testing lr__C ∈ {0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0}',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E3F2FD', edgecolor=BLUE, alpha=0.9))

plt.suptitle('Finding A — Class Imbalance  →  Justifies lr__C Parameter Range',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()

print(f'Positive : {pcts[0]:.1f}%  Negative : {pcts[1]:.1f}%  Neutral : {pcts[2]:.1f}%')
print(f'Imbalance ratio : {pcts[0]/pcts[2]:.1f} : {pcts[1]/pcts[2]:.1f} : 1')

#### Summary Length → `tfidf__ngram_range`

In [ ]:
wc_all = X_train.apply(lambda x: len(str(x).split()))
wc_pos = X_train[y_train == 'positive'].apply(lambda x: len(str(x).split()))
wc_neg = X_train[y_train == 'negative'].apply(lambda x: len(str(x).split()))
wc_neu = X_train[y_train == 'neutral' ].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('white')

cap = 20
for wc, lbl, col in [(wc_pos, 'Positive', '#4CAF50'),
                      (wc_neg, 'Negative', '#F44336'),
                      (wc_neu, 'Neutral',  '#FF9800')]:
    axes[0].hist(wc.clip(upper=cap), bins=range(1, cap + 2),
                 alpha=0.6, label=f'{lbl} (μ={wc.mean():.1f})',
                 color=col, edgecolor='white', density=True)
axes[0].axvline(wc_all.mean(), color='black', linestyle='--',
                linewidth=1.5, label=f'Overall mean={wc_all.mean():.1f}')
axes[0].set_title('Word Count Distribution by Class\n(capped at 20)', fontweight='bold')
axes[0].set_xlabel('Words per Summary')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

cls_names = ['Positive', 'Negative', 'Neutral']
means     = [wc_pos.mean(),   wc_neg.mean(),   wc_neu.mean()]
medians   = [wc_pos.median(), wc_neg.median(), wc_neu.median()]
x_b = np.arange(3)
w_b = 0.35
axes[1].bar(x_b - w_b/2, means,   w_b, color=['#4CAF50','#F44336','#FF9800'],
            label='Mean',   alpha=0.85, edgecolor='white')
axes[1].bar(x_b + w_b/2, medians, w_b, color=['#4CAF50','#F44336','#FF9800'],
            label='Median', alpha=0.45, edgecolor='white', hatch='//')
for i, (m, med) in enumerate(zip(means, medians)):
    axes[1].text(i - w_b/2, m   + 0.05, f'{m:.1f}', ha='center', fontsize=9, fontweight='bold')
    axes[1].text(i + w_b/2, med + 0.05, f'{med:.0f}', ha='center', fontsize=9)
axes[1].set_xticks(x_b)
axes[1].set_xticklabels(cls_names)
axes[1].set_title('Mean vs Median Word Count\nper Sentiment Class', fontweight='bold')
axes[1].set_ylabel('Words per Summary')
axes[1].legend(fontsize=9)
axes[1].set_facecolor(BG)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

thresholds_b = [2, 3, 4, 5, 6, 7, 8]
for wc, lbl, col, lw in [(wc_all, 'All',      'black',    2.5),
                           (wc_pos, 'Positive', '#4CAF50', 1.5),
                           (wc_neg, 'Negative', '#F44336', 1.5),
                           (wc_neu, 'Neutral',  '#FF9800', 1.5)]:
    axes[2].plot(thresholds_b, [(wc <= t).mean() * 100 for t in thresholds_b],
                 marker='o', linewidth=lw, label=lbl, color=col,
                 linestyle='-' if lbl == 'All' else '--')
axes[2].set_xlabel('Word Count Threshold')
axes[2].set_ylabel('% of Summaries ≤ N Words')
axes[2].set_title('% Summaries at Each Length\n(Too Short for Bigram Context)', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].set_facecolor(BG)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

pct_le2 = (wc_all <= 2).mean() * 100
pct_le3 = (wc_all <= 3).mean() * 100
fig.text(0.5, 0.01,
    f'Mean {wc_all.mean():.1f} words  |  {pct_le2:.1f}% summaries ≤ 2 words  |  '
    f'{pct_le3:.1f}% ≤ 3 words\n'
    'Bigrams capture negation phrases ("not good", "very bad") that unigrams split and lose\n'
    'Justifies testing ngram_range ∈ {(1,1), (1,2)}',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', edgecolor='#2E7D32', alpha=0.9))

plt.suptitle('Finding B — Summary Length  →  Justifies tfidf__ngram_range Parameter',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

print(f'Mean words : {wc_all.mean():.1f}')
print(f'≤ 2 words  : {pct_le2:.1f}%')
print(f'≤ 3 words  : {pct_le3:.1f}%')

#### Document Frequency

In [ ]:
thresholds_c = [1, 2, 3]
vocab_kept   = [(doc_freq_base >= t).sum() for t in thresholds_c]
vocab_removed = [total_vocab - k for k in vocab_kept]
pct_kept     = [k / total_vocab * 100 for k in vocab_kept]
pct_removed  = [r / total_vocab * 100 for r in vocab_removed]
n_single     = (doc_freq_base == 1).sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('white')

# Left: vocabulary distribution (log scale)
df_log_bins = np.logspace(0, np.log10(doc_freq_base.max()), 40)
axes[0].hist(doc_freq_base, bins=df_log_bins,
             color=BLUE, alpha=0.85, edgecolor='white')
axes[0].axvline(x=2, color='red', linestyle='--', linewidth=2,
                label='min_df=2 threshold')
axes[0].axvline(x=3, color='orange', linestyle='--', linewidth=2,
                label='min_df=3 threshold')
axes[0].set_xscale('log')
axes[0].set_title('Document Frequency Distribution\n(log scale)', fontweight='bold')
axes[0].set_xlabel('Times Term Appears (log scale)')
axes[0].set_ylabel('Number of Terms')
axes[0].legend(fontsize=9)
axes[0].text(0.98, 0.95,
    f'{n_single:,} terms\nappear only once\n({n_single/total_vocab*100:.0f}% of vocab)',
    transform=axes[0].transAxes, ha='right', va='top', fontsize=9, fontweight='bold',
    bbox=dict(boxstyle='round', facecolor='#FFECB3', edgecolor='#F57F17'))
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Middle: kept vs removed at each threshold
x_c = np.arange(len(thresholds_c))
w_c = 0.38
b_kept = axes[1].bar(x_c - w_c/2, pct_kept,    w_c, label='Kept',    color=BLUE,      edgecolor='white')
b_rmvd = axes[1].bar(x_c + w_c/2, pct_removed, w_c, label='Removed', color='#EF5350', edgecolor='white')
for bar, pct in zip(b_kept,  pct_kept):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{pct:.0f}%', ha='center', fontsize=9, fontweight='bold')
for bar, pct in zip(b_rmvd, pct_removed):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{pct:.0f}%', ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(x_c)
axes[1].set_xticklabels([f'min_df={t}' for t in thresholds_c])
axes[1].set_title('Vocabulary Retained vs Removed\nby min_df Threshold', fontweight='bold')
axes[1].set_ylabel('% of Total Vocabulary')
axes[1].legend(fontsize=10)
axes[1].set_facecolor(BG)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# Right: absolute vocabulary size
bar_cols_c = ['#42A5F5', '#FFA726', '#EF5350']
b_abs = axes[2].bar([f'min_df={t}' for t in thresholds_c], vocab_kept,
                    color=bar_cols_c, edgecolor='white', width=0.5)
for bar, k in zip(b_abs, vocab_kept):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 100, f'{k:,}',
                 ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('Absolute Vocabulary Size\nat each min_df Value', fontweight='bold')
axes[2].set_ylabel('Vocabulary Size')
axes[2].set_xlabel('min_df value')
axes[2].set_facecolor(BG)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

fig.text(0.5, 0.01,
    f'{n_single:,} terms ({n_single/total_vocab*100:.0f}% of vocab) appear in only 1 document  |  '
    f'min_df=2 removes {vocab_removed[1]:,} terms ({pct_removed[1]:.0f}%)\n'
    'Single-occurrence terms carry no generalisable signal for unseen documents\n'
    'Justifies testing min_df ∈ {1, 2, 3}',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FCE4EC', edgecolor='#C62828', alpha=0.9))

plt.suptitle('Finding C — Document Frequency  →  Justifies tfidf__min_df Parameter',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

for t, k in zip(thresholds_c, vocab_kept):
    print(f'min_df={t} : keeps {k:,} terms, removes {total_vocab-k:,}')


#### Term Frequency Skew → `tfidf__sublinear_tf`

In [ ]:
from scipy.stats import skew as scipy_skew

all_text  = ' '.join(X_train)
all_words = all_text.split()
freq_d    = Counter(all_words)
frequencies = sorted(freq_d.values(), reverse=True)
words_list  = [w for w, _ in Counter(all_words).most_common(20)]
raw_tf      = [freq_d[w] for w in words_list]
log_tf      = [np.log1p(f) for f in raw_tf]
med_raw     = np.median(frequencies)
skewness    = scipy_skew(frequencies)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('white')

# Left: raw TF distribution (log scale)
axes[0].hist(frequencies, bins=100, color='#EF5350', alpha=0.85, edgecolor='white', log=True)
axes[0].axvline(x=med_raw, color='black', linestyle='--', linewidth=2,
                label=f'Median = {int(med_raw)}')
axes[0].axvline(x=frequencies[0], color='red', linestyle=':', linewidth=2,
                label=f'Max = {frequencies[0]:,}')
axes[0].set_title('Raw Term Frequency Distribution\n(log y-scale)', fontweight='bold')
axes[0].set_xlabel('Raw Term Frequency')
axes[0].set_ylabel('Number of Terms (log)')
axes[0].legend(fontsize=9)
axes[0].text(0.98, 0.95, f'Skewness = {skewness:.1f}',
             transform=axes[0].transAxes, ha='right', va='top',
             fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='#FFECB3', edgecolor='#F57F17'))
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Middle: raw vs log TF top 20
x_d  = np.arange(len(words_list))
w_d  = 0.38
ax_r = axes[1]
ax_l = ax_r.twinx()
ax_r.bar(x_d - w_d/2, raw_tf, w_d, color='#EF5350', alpha=0.8, label='Raw TF',    edgecolor='white')
ax_l.bar(x_d + w_d/2, log_tf, w_d, color='#42A5F5', alpha=0.8, label='log(1+TF)', edgecolor='white')
ax_r.set_xticks(x_d)
ax_r.set_xticklabels(words_list, rotation=45, ha='right', fontsize=8)
ax_r.set_title('Raw TF vs log(1+TF)\nTop 20 Words', fontweight='bold')
ax_r.set_ylabel('Raw TF',    color='#EF5350')
ax_l.set_ylabel('log(1+TF)', color='#42A5F5')
ax_r.legend(loc='upper left',  fontsize=9)
ax_l.legend(loc='upper right', fontsize=9)
ax_r.set_facecolor(BG)

# Right: dominance ratio
top_n     = [1, 5, 10, 50, 100, 500, 1000]
raw_top   = [frequencies[i - 1] for i in top_n]
log_top   = [np.log1p(f) for f in raw_top]
med_log   = np.log1p(med_raw)
ratio_raw = [f / med_raw for f in raw_top]
ratio_log = [f / med_log for f in log_top]
axes[2].semilogy(top_n, ratio_raw, 'o-', color='#EF5350', linewidth=2,
                  markersize=7, label='Raw TF ratio')
axes[2].semilogy(top_n, ratio_log, 's-', color='#42A5F5', linewidth=2,
                  markersize=7, label='log(1+TF) ratio')
axes[2].axhline(y=1, color='grey', linestyle=':', alpha=0.7)
axes[2].set_title('Dominance Ratio: Top-N vs Median\n(Lower = More Balanced)', fontweight='bold')
axes[2].set_xlabel('Top-N Term Rank')
axes[2].set_ylabel('Freq / Median (log scale)')
axes[2].legend(fontsize=9)
axes[2].set_xticks(top_n)
axes[2].set_facecolor(BG)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

fig.text(0.5, 0.01,
    f'Skewness = {skewness:.1f}  |  Top term: {frequencies[0]:,}  |  '
    f'Median: {int(med_raw)}  |  Top/Median ratio: {int(frequencies[0]/med_raw):,}×\n'
    'log(1+TF) compresses this extreme gap — balances feature contributions\n'
    'Justifies testing sublinear_tf ∈ {True, False}',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E1', edgecolor='#F57F17', alpha=0.9))

plt.suptitle('Finding D — TF Skew  →  Justifies tfidf__sublinear_tf Parameter',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

print(f'Skewness           : {skewness:.1f}')
print(f'Top term frequency : {frequencies[0]:,}')
print(f'Median frequency   : {int(med_raw)}')
print(f'Top / Median ratio : {int(frequencies[0]/med_raw):,}×')

#### Cross-Class Vocabulary Overlap → `lr__penalty`

In [ ]:
N_e          = 50
top_words_e  = {}
word_freqs_e = {}
for label in ['positive', 'negative', 'neutral']:
    mask = y_train == label
    wds  = ' '.join(X_train[mask]).split()
    freq = Counter(wds)
    top_words_e[label]  = set([w for w, _ in freq.most_common(N_e)])
    word_freqs_e[label] = freq

pos_e       = top_words_e['positive']
neg_e       = top_words_e['negative']
neu_e       = top_words_e['neutral']
pos_neg_e   = pos_e & neg_e
pos_neu_e   = pos_e & neu_e
neg_neu_e   = neg_e & neu_e
all_three_e = pos_e & neg_e & neu_e       # used later in Validation 1

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.patch.set_facecolor('white')

overlap_labels = ['Pos ∩ Neg', 'Pos ∩ Neu', 'Neg ∩ Neu', 'All 3 Classes']
overlap_vals   = [len(pos_neg_e), len(pos_neu_e), len(neg_neu_e), len(all_three_e)]
overlap_colors = ['#AB47BC', '#EF5350', '#FF7043', '#5C6BC0']
bars_e = axes[0].bar(overlap_labels, overlap_vals,
                      color=overlap_colors, edgecolor='white', linewidth=1.5)
axes[0].set_title(f'Shared Words in Top {N_e} Terms\nPer Sentiment Class', fontweight='bold')
axes[0].set_ylabel('Number of Shared Words')
axes[0].set_ylim(0, N_e * 0.75)
for bar, val in zip(bars_e, overlap_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.2,
                 f'{val}\n({val/N_e*100:.0f}% of top {N_e})',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

shared_pn = sorted(list(pos_neu_e),
                    key=lambda w: (word_freqs_e['positive'][w] +
                                   word_freqs_e['neutral'][w]),
                    reverse=True)[:12]
x_e = np.arange(len(shared_pn))
w_e = 0.35
axes[1].bar(x_e - w_e/2, [word_freqs_e['positive'][wd] for wd in shared_pn],
            w_e, label='Positive', color='#4CAF50', alpha=0.85, edgecolor='white')
axes[1].bar(x_e + w_e/2, [word_freqs_e['neutral'][wd]  for wd in shared_pn],
            w_e, label='Neutral',  color='#FF9800', alpha=0.85, edgecolor='white')
axes[1].set_xticks(x_e)
axes[1].set_xticklabels(shared_pn, rotation=45, ha='right', fontsize=9)
axes[1].set_title('Top Shared Words:\nPositive vs Neutral (Hardest Pair)', fontweight='bold')
axes[1].set_ylabel('Frequency in Training Set')
axes[1].legend(fontsize=10)
axes[1].set_facecolor(BG)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

shared_all = sorted(list(all_three_e),
                     key=lambda w: sum(word_freqs_e[c][w]
                                       for c in ['positive','negative','neutral']),
                     reverse=True)[:10]
x_e3 = np.arange(len(shared_all))
w_e3 = 0.25
axes[2].bar(x_e3 - w_e3, [word_freqs_e['positive'][w] for w in shared_all],
            w_e3, label='Positive', color='#4CAF50', alpha=0.85, edgecolor='white')
axes[2].bar(x_e3,         [word_freqs_e['negative'][w] for w in shared_all],
            w_e3, label='Negative', color='#F44336', alpha=0.85, edgecolor='white')
axes[2].bar(x_e3 + w_e3, [word_freqs_e['neutral'][w]  for w in shared_all],
            w_e3, label='Neutral',  color='#FF9800', alpha=0.85, edgecolor='white')
axes[2].set_xticks(x_e3)
axes[2].set_xticklabels(shared_all, rotation=45, ha='right', fontsize=9)
axes[2].set_title('Words Shared Across\nAll 3 Classes', fontweight='bold')
axes[2].set_ylabel('Frequency in Training Set')
axes[2].legend(fontsize=9)
axes[2].set_facecolor(BG)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

fig.text(0.5, 0.01,
    f'{len(all_three_e)} of top-{N_e} words appear in ALL 3 classes  |  '
    f'{len(pos_neu_e)} shared Positive & Neutral  |  '
    f'{len(pos_neg_e)} shared Positive & Negative\n'
    'l1 penalty zeros ambiguous shared features — l2 distributes weight across all of them\n'
    'Justifies testing lr__penalty ∈ {l1, l2} with solvers lbfgs (l2) and saga (l1)',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FCE4EC', edgecolor='#C62828', alpha=0.9))

plt.suptitle('Finding E — Vocabulary Overlap  →  Justifies lr__penalty Parameter (l1 vs l2)',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

print(f'All 3 classes shared : {len(all_three_e)}')
print(f'Positive ∩ Neutral   : {len(pos_neu_e)}')
print(f'Positive ∩ Negative  : {len(pos_neg_e)}')

#### C Parameter Sensitivity → `lr__C` Range Confirmation

In [ ]:
np.random.seed(42)
idx_f    = np.random.choice(len(X_train), 8000, replace=False)
X_f_samp = X_train.iloc[idx_f].values
y_f_samp = y_train.iloc[idx_f].values

# Pre-vectorise the sample — solver=lbfgs used here for consistency with
# the final model (l2 + lbfgs is the fixed choice from Finding E)
tfidf_f  = TfidfVectorizer(max_features=20000, sublinear_tf=True)
X_f_vec  = tfidf_f.fit_transform(X_f_samp)

C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
mean_f1  = []
std_f1   = []

print('5-fold CV across C values on 8k sample (lbfgs solver)...')
for C in C_values:
    model  = LogisticRegression(C=C, class_weight='balanced',
                                random_state=42, max_iter=500, solver='lbfgs')
    scores = cross_val_score(model, X_f_vec, y_f_samp,
                             cv=5, scoring='f1_macro')
    mean_f1.append(scores.mean())
    std_f1.append(scores.std())
    print(f'  C={C:<8} → Macro F1 = {scores.mean():.4f} ± {scores.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('white')

x_f = range(len(C_values))
axes[0].plot(x_f, mean_f1, 'o-', color=BLUE, linewidth=2.5, markersize=9, zorder=3)
axes[0].fill_between(x_f,
                      [m - s for m, s in zip(mean_f1, std_f1)],
                      [m + s for m, s in zip(mean_f1, std_f1)],
                      alpha=0.2, color=BLUE, label='±1 std')
axes[0].set_xticks(x_f)
axes[0].set_xticklabels([str(c) for c in C_values])
axes[0].set_title('5-Fold CV Macro F1 vs C\n(8k Sample — Range Confirmation, lbfgs)', fontweight='bold')
axes[0].set_xlabel('C Value')
axes[0].set_ylabel('Macro F1-Score')
best_c_idx = int(np.argmax(mean_f1))
axes[0].axvline(x=best_c_idx, color='red', linestyle='--', alpha=0.7)
axes[0].annotate(f'Sample best: C={C_values[best_c_idx]}\nF1={mean_f1[best_c_idx]:.4f}',
                 (best_c_idx, mean_f1[best_c_idx]),
                 xytext=(best_c_idx + 0.4, mean_f1[best_c_idx] - 0.008),
                 fontsize=9, fontweight='bold', color='red',
                 arrowprops=dict(arrowstyle='->', color='red'))
axes[0].legend(fontsize=9)
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].axis('off')
td_f = [[str(c), f'{m:.4f}', f'{s:.4f}',
         '← Sample best' if i == best_c_idx else '']
        for i, (c, m, s) in enumerate(zip(C_values, mean_f1, std_f1))]
t_f = axes[1].table(cellText=td_f,
                     colLabels=['C Value', 'Mean F1', 'Std', 'Note'],
                     loc='center', cellLoc='center')
t_f.auto_set_font_size(False); t_f.set_fontsize(10.5); t_f.scale(1.3, 1.8)
for j in range(4):
    t_f[0, j].set_facecolor(BLUE)
    t_f[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(C_values) + 1):
    fc = '#FFF9C4' if i - 1 == best_c_idx else ('#E8F5E9' if i % 2 == 0 else '#FFFFFF')
    for j in range(4):
        t_f[i, j].set_facecolor(fc)
axes[1].set_title('C Sensitivity — Sample CV Results', fontweight='bold', pad=15)

fig.text(0.5, 0.01,
    'Sample CV confirms F1 varies meaningfully across the C range — '
    'peak is around C=1.0, drop-off beyond C=10\n'
    'Exact best C is determined on full training data in Part 3\n'
    'Justifies testing lr__C ∈ {0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0}',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E3F2FD', edgecolor=BLUE, alpha=0.9))

plt.suptitle('Finding F — C Sensitivity  →  Confirms lr__C Parameter Range',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

#### Vocabulary Coverage → `tfidf__max_features`

In [ ]:
sorted_df_g = np.sort(doc_freq_base)[::-1]
cumul_df_g  = np.cumsum(sorted_df_g)
total_td_g  = cumul_df_g[-1]

feature_caps = [5000, 10000, 15000, 20000, 25000, 30000, total_vocab]
cap_labels   = ['5k', '10k', '15k', '20k', '25k', '30k', f'All\n({total_vocab:,})']
coverages    = [cumul_df_g[min(cap, total_vocab) - 1] / total_td_g * 100
                for cap in feature_caps]

# Sparsity: refit only for the capped values (full vocab uses X_base)
sparsities = []
for cap in feature_caps:
    if cap >= total_vocab:
        sp = (1 - X_base.nnz / (X_base.shape[0] * X_base.shape[1])) * 100
    else:
        vt = TfidfVectorizer(max_features=cap)
        Xt = vt.fit_transform(X_train)
        sp = (1 - Xt.nnz / (Xt.shape[0] * Xt.shape[1])) * 100
    sparsities.append(sp)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('white')

bar_cols_g = ['#EF5350' if i < 2 else '#FFA726' if i < 4 else '#42A5F5'
              for i in range(len(feature_caps))]
bars_g = axes[0].bar(cap_labels, coverages, color=bar_cols_g, edgecolor='white', linewidth=1.3)
axes[0].set_title('Term-Document Coverage\nvs max_features Cap', fontweight='bold')
axes[0].set_xlabel('max_features')
axes[0].set_ylabel('% of Term-Document Occurrences Covered')
axes[0].set_ylim(0, 110)
for bar, cov in zip(bars_g, coverages):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, f'{cov:.1f}%',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].plot(range(len(feature_caps)), sparsities, 'o-', color=BLUE,
             linewidth=2.5, markersize=9)
axes[1].set_xticks(range(len(feature_caps)))
axes[1].set_xticklabels(cap_labels)
axes[1].set_title('Matrix Sparsity\nat each max_features Cap', fontweight='bold')
axes[1].set_xlabel('max_features')
axes[1].set_ylabel('Sparsity (%)')
axes[1].set_ylim(99.85, 100.02)
for i, sp in enumerate(sparsities):
    axes[1].annotate(f'{sp:.4f}%', (i, sp),
                     textcoords='offset points', xytext=(0, 6),
                     ha='center', fontsize=8.5)
axes[1].fill_between(range(len(feature_caps)), sparsities, 99.85, alpha=0.1, color=BLUE)
axes[1].set_facecolor(BG)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

fig.text(0.5, 0.01,
    f'Top 30k terms cover {coverages[5]:.1f}% of occurrences  |  '
    f'Full vocab ({total_vocab:,}) covers 100.0%\n'
    'Capping at 30k reduces noise from rare terms with minimal coverage loss\n'
    'Justifies testing max_features ∈ {30000, None}',
    ha='center', fontsize=10, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8EAF6', edgecolor=BLUE, alpha=0.9))

plt.suptitle('Finding G — Vocabulary Coverage  →  Justifies tfidf__max_features Parameter',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

print(f'Total vocabulary : {total_vocab:,}')
for cap, cov in zip(feature_caps, coverages):
    print(f'  max_features={cap:<6}: {cov:.1f}% coverage')

---
### Part 2 — Parameter Validation
#### Validation 1 — `tfidf__max_df` (Rejected)

In [ ]:
vocab_arr = tfidf_base.get_feature_names_out()
df_pct_v1 = dict(zip(vocab_arr, doc_freq_base / total_docs * 100))

conf_words = sorted(list(all_three_e),
                     key=lambda w: df_pct_v1.get(w, 0), reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('white')

df_vals = [df_pct_v1.get(w, 0) for w in conf_words]
axes[0].barh(conf_words[::-1], df_vals[::-1],
             color='#EF5350', alpha=0.85, edgecolor='white')
for thresh, col, lbl in [(70, 'orange', 'max_df=0.70'),
                          (85, 'red',    'max_df=0.85')]:
    axes[0].axvline(x=thresh, color=col, linestyle='--', alpha=0.9, label=lbl)
axes[0].set_title('Document Frequency of\nCross-Class Overlap Words', fontweight='bold')
axes[0].set_xlabel('% of Training Documents Containing Word')
axes[0].legend(fontsize=9)
for i, (w, val) in enumerate(zip(conf_words[::-1], df_vals[::-1])):
    axes[0].text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=8)
axes[0].text(0.98, 0.05,
    '⚠ No word exceeds 33%\nAll thresholds remove\nZERO terms',
    transform=axes[0].transAxes, ha='right', fontsize=9, fontweight='bold',
    bbox=dict(boxstyle='round', facecolor='#FFECB3', edgecolor='#F57F17'))
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

thresholds_v1 = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
removed_v1    = [(doc_freq_base > t * total_docs).sum() for t in thresholds_v1]
axes[1].bar([str(t) for t in thresholds_v1], removed_v1,
            color=BLUE, alpha=0.85, edgecolor='white')
axes[1].set_title('Terms Removed\nat each max_df Threshold', fontweight='bold')
axes[1].set_xlabel('max_df Threshold')
axes[1].set_ylabel('Number of Terms Removed')
for bar, val in zip(axes[1].patches, removed_v1):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3, str(val),
                 ha='center', fontsize=10, fontweight='bold')
axes[1].set_facecolor(BG)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Validation 1 — max_df  →  REJECTED: No Word Exceeds Any Useful Threshold',
             fontsize=12, fontweight='bold', color='#B71C1C', y=1.01)
plt.tight_layout()
plt.show()

print(f'Highest word doc frequency : {max(df_pct_v1.values()):.1f}%')
for t in [0.5, 0.7, 0.85, 0.9]:
    n = (doc_freq_base > t * total_docs).sum()
    print(f'max_df={t} : removes {n} terms')
print('DECISION: max_df excluded from grid — ineffective on this dataset')

#### Validation 2 — Custom Domain Stopwords and SelectKBest chi²

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

domain_stopwords = [
    'money', 'one', 'also', 'sound', 'use', 'much', 'using',
    'installation', 'got', 'battery', 'time', 'like', 'go'
]
combined_stops = list(ENGLISH_STOP_WORDS) + domain_stopwords

np.random.seed(42)
idx_v2 = np.random.choice(len(X_train), 8000, replace=False)
X_v2   = X_train.iloc[idx_v2].values
y_v2   = y_train.iloc[idx_v2].values

def make_val_pipe(use_sw=False, k=None, C=1.0):
    stop  = combined_stops if use_sw else 'english'
    steps = [('tfidf', TfidfVectorizer(
                  max_features=20000, sublinear_tf=True, stop_words=stop))]
    if k:
        steps.append(('select', SelectKBest(chi2, k=k)))
    steps.append(('lr', LogisticRegression(
        C=C, class_weight='balanced', solver='lbfgs',
        random_state=42, max_iter=500)))
    return Pipeline(steps)

configs_v2 = [
    ('1. Baseline',                        False, None,  1.0),
    ('2. + Custom Domain Stopwords',       True,  None,  1.0),
    ('3. + SelectKBest k=5,000',           False, 5000,  1.0),
    ('4. + SelectKBest k=10,000',          False, 10000, 1.0),
    ('5. Custom SW + SelectKBest k=5,000', True,  5000,  1.0),
    ('6. Custom SW + SelectKBest k=10k',   True,  10000, 1.0),
]

results_v2 = []
print('Running 5-fold CV (8k sample)...')
for name, use_sw, k, C in configs_v2:
    pipe  = make_val_pipe(use_sw, k, C)
    score = cross_val_score(pipe, X_v2, y_v2, cv=5, scoring='f1_macro')
    results_v2.append((name, score.mean(), score.std()))
    delta = score.mean() - results_v2[0][1] if len(results_v2) > 1 else 0.0
    print(f'  {name:<40} F1={score.mean():.4f} ({delta:+.4f})')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('white')

names_v2    = [r[0] for r in results_v2]
means_v2    = [r[1] for r in results_v2]
stds_v2     = [r[2] for r in results_v2]
baseline_v2 = means_v2[0]
bar_col_v2  = ['#90A4AE' if i == 0 else
                '#4CAF50' if m >= baseline_v2 else '#EF5350'
                for i, m in enumerate(means_v2)]

axes[0].barh(range(len(names_v2)), means_v2,
             xerr=stds_v2, color=bar_col_v2, edgecolor='white',
             linewidth=1.3, capsize=4, height=0.6)
axes[0].axvline(x=baseline_v2, color='black', linestyle='--',
                linewidth=1.5, label=f'Baseline F1={baseline_v2:.4f}')
axes[0].set_yticks(range(len(names_v2)))
axes[0].set_yticklabels(names_v2, fontsize=9)
axes[0].set_title('Impact on Macro F1\n(5-Fold CV, 8k Sample)', fontweight='bold')
axes[0].set_xlabel('Macro F1 Score')
axes[0].legend(fontsize=9)
for i, (m, s) in enumerate(zip(means_v2, stds_v2)):
    delta = m - baseline_v2
    axes[0].text(m + s + 0.001, i,
                 f'{m:.4f} ({delta:+.4f})',
                 va='center', fontsize=8.5, fontweight='bold',
                 color='#1B5E20' if delta > 0 else '#B71C1C' if delta < 0 else 'black')
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].set_xlim(min(means_v2) - 0.05, max(means_v2) + 0.07)

axes[1].axis('off')
td_v2 = []
for i, (name, m, s) in enumerate(results_v2):
    delta   = m - baseline_v2
    verdict = '← Baseline' if i == 0 else ('✓ Helps' if delta > 0 else '✗ Hurts')
    td_v2.append([name.split('.')[1].strip() if '.' in name else name,
                  f'{m:.4f}', f'{delta:+.4f}', verdict])
t_v2 = axes[1].table(cellText=td_v2,
                      colLabels=['Technique', 'Macro F1', 'Δ', 'Verdict'],
                      loc='center', cellLoc='center')
t_v2.auto_set_font_size(False); t_v2.set_fontsize(10); t_v2.scale(1.1, 1.7)
for j in range(4):
    t_v2[0, j].set_facecolor(BLUE)
    t_v2[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(results_v2) + 1):
    fc = '#FFFFFF' if i % 2 == 0 else '#F5F5F5'
    if i == 1:                         fc = '#E3F2FD'
    elif results_v2[i-1][1] < baseline_v2: fc = '#FFEBEE'
    for j in range(4):
        t_v2[i, j].set_facecolor(fc)
axes[1].set_title('Validation Results Summary', fontweight='bold', pad=15)

plt.suptitle('Validation 2 — Custom Stopwords & SelectKBest  →  BOTH REJECTED',
             fontsize=12, fontweight='bold', color='#B71C1C', y=1.01)
plt.tight_layout()
plt.show()

print('\nCONCLUSION:')
print('  Custom stopwords → hurts F1 — low chi2 words still carry residual signal')
print('  SelectKBest k=5k → hurts badly — aggressive feature removal loses signal')
print('  SelectKBest k=10k → neutral — TF-IDF IDF already handles this')
print('  BOTH excluded from final parameter grid')

#### Validated Parameter Grid Summary

In [ ]:
params_summary = [
    ['tfidf__max_features', 'G',   '30k=99.3% coverage; None=100%',              '✓ GRID',    '[30000, None]'],
    ['tfidf__ngram_range',  'B',   'Mean 6.4 words; 35.9% ≤2 words',             '✓ FIXED',   '(1, 2)'],
    ['tfidf__sublinear_tf', 'D',   'Skewness 81.9; top/median = 39,685×',         '✓ FIXED',   'True'],
    ['tfidf__min_df',       'C',   '62% of vocab in only 1 doc',                  '✓ FIXED',   '1'],
    ['tfidf__max_df',       'V1',  'Max doc freq = 32.1% — thresholds remove ≈0', '✗ REJECTED','Excluded'],
    ['lr__C',               'A+F', 'Peak at 0.5–2.0 confirmed on 8k sample',      '✓ GRID',    '[0.5, 1.0, 1.5, 2.0]'],
    ['lr__penalty+solver',  'E',   'l2+lbfgs wins by +0.060 F1 over l1+saga',     '✓ FIXED',   'l2 + lbfgs'],
    ['Custom Stopwords',    'V2',  'Removal hurts F1 (−0.07)',                    '✗ REJECTED','Excluded'],
    ['SelectKBest chi²',    'V2',  'k=5k hurts; k=10k neutral — IDF covers this', '✗ REJECTED','Excluded'],
]

fig, ax = plt.subplots(figsize=(18, 5.5))
ax.axis('off')
fig.patch.set_facecolor('white')

t_s = ax.table(cellText=params_summary,
               colLabels=['Parameter', 'Finding', 'EDA Evidence', 'Decision', 'Values'],
               loc='center', cellLoc='left',
               colWidths=[0.18, 0.06, 0.32, 0.10, 0.28])
t_s.auto_set_font_size(False); t_s.set_fontsize(9.5); t_s.scale(1, 2.0)

for (r, c), cell in t_s.get_celld().items():
    if r == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color='white', fontweight='bold')
    else:
        row_data = params_summary[r - 1]
        if '✗ REJECTED' in row_data[3]:
            cell.set_facecolor('#FFEBEE')
        elif '✓ GRID' in row_data[3]:
            cell.set_facecolor('#E8F5E9')
        elif r % 2 == 0:
            cell.set_facecolor('#E3F2FD')
        else:
            cell.set_facecolor('#FFFFFF')
        if c == 3:
            color = '#1B5E20' if '✓' in row_data[3] else '#B71C1C'
            cell.set_text_props(fontweight='bold', color=color)
    cell.set_edgecolor('#E0E0E0')

plt.title('Q3 — Validated Parameter Grid (EDA-Driven)\nGreen = in grid  |  Blue = fixed  |  Red = rejected',
          fontsize=13, fontweight='bold', color=DARK, pad=15)
plt.tight_layout()
plt.show()

print('Grid parameters : max_features [30000, None]  ×  C [0.5, 1.0, 1.5, 2.0]')
print('Fixed           : ngram_range=(1,2), sublinear_tf=True, min_df=1, penalty=l2, solver=lbfgs')
print('Rejected        : max_df, custom stopwords, SelectKBest')


---
### Part 3 — GridSearchCV Tuning

In [ ]:
pipeline_q3 = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),    # FIXED — Finding B
        sublinear_tf=True,     # FIXED — Finding D
        min_df=1,              # FIXED — Finding C
    )),
    ('lr', LogisticRegression(
        class_weight='balanced',   # FIXED — Finding A (13.3:1 imbalance)
        random_state=42,
        max_iter=1000,
        penalty='l2',          # FIXED — Finding E (l2 >> l1)
        solver='lbfgs',        # FIXED — paired with l2
    ))
])

param_grid_q3 = {
    'tfidf__max_features' : [30000, None],            # Finding G
    'lr__C'               : [0.5, 1.0, 1.5, 2.0],    # Finding A+F
}

search_q3 = GridSearchCV(
    pipeline_q3,
    param_grid_q3,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

n_combos = len(param_grid_q3['tfidf__max_features']) * len(param_grid_q3['lr__C'])
print('Running GridSearchCV on full training set...')
print(f'  Combinations : {n_combos}  (2 max_features × 4 C values)')
print(f'  CV folds     : 3')
print(f'  Total fits   : {n_combos * 3}')
print(f'  Scoring      : f1_macro')
print()

t_start = time.time()
search_q3.fit(X_train, y_train)
t_elapsed = time.time() - t_start

print(f'\nSearch complete in {t_elapsed/60:.1f} minutes')
print()
print('=' * 55)
print(' BEST PARAMETERS FOUND')
print('=' * 55)
for param, value in search_q3.best_params_.items():
    print(f'  {param:<30}: {value}')
print(f'\n  Best CV Macro F1 : {search_q3.best_score_:.4f}')

#### All Combinations Ranked

In [ ]:
cv_results_df = pd.DataFrame({
    'max_features' : [p['tfidf__max_features'] for p in search_q3.cv_results_['params']],
    'C'            : [p['lr__C']               for p in search_q3.cv_results_['params']],
    'Mean F1'      : search_q3.cv_results_['mean_test_score'],
    'Std'          : search_q3.cv_results_['std_test_score'],
    'Rank'         : search_q3.cv_results_['rank_test_score'],
}).sort_values('Rank').reset_index(drop=True)

print('All combinations ranked by CV Macro F1:')
print(cv_results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')
fig.patch.set_facecolor('white')

table_rows = [
    [str(row['max_features']), str(row['C']),
     f'{row["Mean F1"]:.4f}', f'{row["Std"]:.4f}', str(int(row["Rank"]))]
    for _, row in cv_results_df.iterrows()
]
tbl_r = ax.table(
    cellText=table_rows,
    colLabels=['max_features', 'C', 'Mean CV F1', 'Std', 'Rank'],
    cellLoc='center', loc='center',
    colWidths=[0.22, 0.15, 0.22, 0.18, 0.12]
)
tbl_r.auto_set_font_size(False); tbl_r.set_fontsize(10.5); tbl_r.scale(1, 1.9)

for (r, c), cell in tbl_r.get_celld().items():
    if r == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color='white', fontweight='bold')
    elif r == 1:
        cell.set_facecolor('#FFF9C4')
        cell.set_text_props(fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    else:
        cell.set_facecolor('#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title(
    'GridSearchCV — All Combinations Ranked by CV Macro F1\n'
    '(Fixed: ngram=(1,2), sublinear_tf=True, min_df=1, penalty=l2+lbfgs, cv=3)',
    fontsize=12, fontweight='bold', color=DARK, pad=12
)
plt.tight_layout()
plt.show()

#### Best Parameters — Summary Table

In [ ]:
best_p = search_q3.best_params_

best_params_display = [
    ['tfidf__max_features',  str(best_p['tfidf__max_features']),  'Finding G'],
    ['tfidf__ngram_range',   '(1, 2)  [FIXED]',                  'Finding B'],
    ['tfidf__sublinear_tf',  'True    [FIXED]',                  'Finding D'],
    ['tfidf__min_df',        '1       [FIXED]',                  'Finding C'],
    ['lr__C',                str(best_p['lr__C']),                'Finding A+F'],
    ['lr__penalty',          'l2      [FIXED]',                  'Finding E'],
    ['lr__solver',           'lbfgs   [FIXED]',                  'Paired with l2'],
    ['class_weight',         'balanced [FIXED]',                  'Finding A'],
    ['Best CV Macro F1',     f'{search_q3.best_score_:.4f}',     '3-fold CV on training set'],
]

fig, ax = plt.subplots(figsize=(11, 5))
ax.axis('off')
fig.patch.set_facecolor('white')

tbl = ax.table(
    cellText=best_params_display,
    colLabels=['Parameter', 'Best Value', 'EDA Justification'],
    loc='center', cellLoc='left',
    colWidths=[0.30, 0.28, 0.38]
)
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5); tbl.scale(1, 1.95)

for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color='white', fontweight='bold')
    elif r == len(best_params_display):
        cell.set_facecolor('#FFF9C4')
        cell.set_text_props(fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    else:
        cell.set_facecolor('#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title(
    'GridSearchCV — Best Parameters Found\n'
    f'({n_combos} combinations × 3-fold = {n_combos*3} fits | scoring=f1_macro)',
    fontsize=13, fontweight='bold', color=DARK, pad=12
)
plt.tight_layout()
plt.show()

---
### Part 4 — Tuned Model Evaluation

In [ ]:
y_pred_tuned = search_q3.predict(X_test)

report_tuned  = classification_report(y_test, y_pred_tuned,
                                       target_names=LABELS, output_dict=True)
acc_tuned     = accuracy_score(y_test, y_pred_tuned)
macro_p_t     = precision_score(y_test, y_pred_tuned, average='macro')
macro_r_t     = recall_score(y_test, y_pred_tuned, average='macro')
macro_f1_t    = f1_score(y_test, y_pred_tuned, average='macro')
weighted_f1_t = f1_score(y_test, y_pred_tuned, average='weighted')

print('=== TUNED MODEL — CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred_tuned, labels=LABELS, target_names=LABELS))
print(f'Accuracy          : {acc_tuned:.4f}')
print(f'Macro Precision   : {macro_p_t:.4f}')
print(f'Macro Recall      : {macro_r_t:.4f}')
print(f'Macro F1-Score    : {macro_f1_t:.4f}')
print(f'Weighted F1-Score : {weighted_f1_t:.4f}')

#### Styled Classification Report

In [ ]:
table_data_t = []
for label in LABELS:
    r = report_tuned[label]
    table_data_t.append([
        label.capitalize(),
        f'{r["precision"]:.4f}', f'{r["recall"]:.4f}',
        f'{r["f1-score"]:.4f}',  f'{int(r["support"]):,}'
    ])
table_data_t.append(['Macro Avg',
    f'{report_tuned["macro avg"]["precision"]:.4f}',
    f'{report_tuned["macro avg"]["recall"]:.4f}',
    f'{report_tuned["macro avg"]["f1-score"]:.4f}',
    f'{int(report_tuned["macro avg"]["support"]):,}'])
table_data_t.append(['Weighted Avg',
    f'{report_tuned["weighted avg"]["precision"]:.4f}',
    f'{report_tuned["weighted avg"]["recall"]:.4f}',
    f'{report_tuned["weighted avg"]["f1-score"]:.4f}',
    f'{int(report_tuned["weighted avg"]["support"]):,}'])
table_data_t.append(['Accuracy', '', '', f'{acc_tuned:.4f}', f'{len(y_test):,}'])

fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')
fig.patch.set_facecolor('white')
tbl_t = ax.table(
    cellText=table_data_t,
    colLabels=['Class', 'Precision', 'Recall', 'F1-Score', 'Support'],
    cellLoc='center', loc='center',
    colWidths=[0.22, 0.18, 0.18, 0.18, 0.18]
)
tbl_t.auto_set_font_size(False); tbl_t.set_fontsize(11); tbl_t.scale(1, 2.0)

for (r, c), cell in tbl_t.get_celld().items():
    if r == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color='white', fontweight='bold')
    elif r <= 3 and c == 0:
        cell.set_facecolor(['#4CAF50', '#F44336', '#FF9800'][r - 1])
        cell.set_text_props(color='white', fontweight='bold')
    elif r > 3:
        cell.set_facecolor('#CFD8DC')
        cell.set_text_props(fontweight='bold')
    else:
        cell.set_facecolor('#E3F2FD' if r % 2 == 0 else '#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title('Logistic Regression — Tuned Model Classification Report',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

#### Confusion Matrix

In [ ]:
cm_t = confusion_matrix(y_test, y_pred_tuned, labels=LABELS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues',
            xticklabels=[l.capitalize() for l in LABELS],
            yticklabels=[l.capitalize() for l in LABELS],
            linewidths=0.5, linecolor='white',
            ax=axes[0], annot_kws={'size': 12, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix — Tuned (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontweight='bold')
axes[0].set_ylabel('True Label', fontweight='bold')

cm_t_norm = cm_t.astype(float) / cm_t.sum(axis=1, keepdims=True)
sns.heatmap(cm_t_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=[l.capitalize() for l in LABELS],
            yticklabels=[l.capitalize() for l in LABELS],
            linewidths=0.5, linecolor='white',
            ax=axes[1], annot_kws={'size': 12, 'weight': 'bold'})
axes[1].set_title('Confusion Matrix — Tuned (Normalised)', fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontweight='bold')
axes[1].set_ylabel('True Label', fontweight='bold')

plt.suptitle('Logistic Regression — Tuned Model Confusion Matrix',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Per-Class Metric Bar Chart

In [ ]:
metrics_tuned = {
    'Precision': [report_tuned[l]['precision'] for l in LABELS],
    'Recall'   : [report_tuned[l]['recall']    for l in LABELS],
    'F1-Score' : [report_tuned[l]['f1-score']  for l in LABELS]
}

x_t     = np.arange(len(LABELS))
width_t = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
bars_t1 = ax.bar(x_t - width_t, metrics_tuned['Precision'], width_t,
                  label='Precision', color='#1565C0', edgecolor='white')
bars_t2 = ax.bar(x_t,             metrics_tuned['Recall'],    width_t,
                  label='Recall',    color='#42A5F5', edgecolor='white')
bars_t3 = ax.bar(x_t + width_t,   metrics_tuned['F1-Score'],  width_t,
                  label='F1-Score',  color='#90CAF9', edgecolor='white')

for bars_t in [bars_t1, bars_t2, bars_t3]:
    for bar in bars_t:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x_t)
ax.set_xticklabels([l.capitalize() for l in LABELS], fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Logistic Regression — Tuned Model Per-Class Precision, Recall and F1-Score',
             fontweight='bold')
ax.legend(fontsize=11)
ax.set_facecolor(BG)
plt.tight_layout()
plt.show()

---
### Part 5 — Baseline vs Tuned Comparison

In [ ]:
f1_base_per  = [report[l]['f1-score']       for l in LABELS]
f1_tuned_per = [report_tuned[l]['f1-score'] for l in LABELS]

comparison = [
    ['Accuracy',    f'{accuracy:.4f}',    f'{acc_tuned:.4f}',
     f'{acc_tuned - accuracy:+.4f}'],
    ['Macro F1',    f'{macro_f1:.4f}',    f'{macro_f1_t:.4f}',
     f'{macro_f1_t - macro_f1:+.4f}'],
    ['Weighted F1', f'{weighted_f1:.4f}', f'{weighted_f1_t:.4f}',
     f'{weighted_f1_t - weighted_f1:+.4f}'],
    ['Positive F1', f'{f1_base_per[0]:.4f}', f'{f1_tuned_per[0]:.4f}',
     f'{f1_tuned_per[0] - f1_base_per[0]:+.4f}'],
    ['Negative F1', f'{f1_base_per[1]:.4f}', f'{f1_tuned_per[1]:.4f}',
     f'{f1_tuned_per[1] - f1_base_per[1]:+.4f}'],
    ['Neutral F1',  f'{f1_base_per[2]:.4f}', f'{f1_tuned_per[2]:.4f}',
     f'{f1_tuned_per[2] - f1_base_per[2]:+.4f}'],
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('white')

metric_names = [r[0] for r in comparison]
base_vals    = [float(r[1]) for r in comparison]
tuned_vals   = [float(r[2]) for r in comparison]
x_cmp = np.arange(len(metric_names))
w_cmp = 0.35
b_cmp1 = axes[0].bar(x_cmp - w_cmp/2, base_vals,  w_cmp,
                      label='Baseline', color='#90A4AE', edgecolor='white')
b_cmp2 = axes[0].bar(x_cmp + w_cmp/2, tuned_vals, w_cmp,
                      label='Tuned',    color=BLUE,     edgecolor='white')
for bar, val in zip(b_cmp1, base_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.005, f'{val:.4f}',
                 ha='center', fontsize=7.5, rotation=45)
for bar, val in zip(b_cmp2, tuned_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.005, f'{val:.4f}',
                 ha='center', fontsize=7.5, rotation=45, fontweight='bold')
axes[0].set_xticks(x_cmp)
axes[0].set_xticklabels(metric_names, fontsize=10)
axes[0].set_title('Baseline vs Tuned — All Metrics', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.15)
axes[0].legend(fontsize=10)
axes[0].set_facecolor(BG)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

deltas       = [float(r[3]) for r in comparison]
delta_colors = ['#4CAF50' if d > 0 else '#F44336' for d in deltas]
axes[1].bar(metric_names, deltas, color=delta_colors, edgecolor='white', linewidth=1.3)
axes[1].axhline(y=0, color='black', linewidth=1)
for i, d in enumerate(deltas):
    axes[1].text(i, d + (0.001 if d >= 0 else -0.004),
                 f'{d:+.4f}', ha='center', fontsize=9, fontweight='bold',
                 color='#1B5E20' if d > 0 else '#B71C1C')
axes[1].set_title('Improvement per Metric\n(Tuned − Baseline)', fontweight='bold')
axes[1].set_ylabel('Δ Score')
axes[1].set_facecolor(BG)
axes[1].tick_params(axis='x', rotation=20)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Logistic Regression — Baseline vs Tuned Comparison',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)
plt.tight_layout()
plt.show()

#### Styled Comparison Table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
fig.patch.set_facecolor('white')

tbl_cmp = ax.table(
    cellText=comparison,
    colLabels=['Metric', 'Baseline', 'Tuned', 'Δ Change'],
    cellLoc='center', loc='center',
    colWidths=[0.28, 0.22, 0.22, 0.22]
)
tbl_cmp.auto_set_font_size(False); tbl_cmp.set_fontsize(11); tbl_cmp.scale(1, 2.0)

for (r, c), cell in tbl_cmp.get_celld().items():
    if r == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color='white', fontweight='bold')
    elif c == 3 and r > 0:
        dval = float(comparison[r - 1][3])
        cell.set_facecolor('#E8F5E9' if dval > 0 else '#FFEBEE')
        cell.set_text_props(fontweight='bold',
                             color='#1B5E20' if dval > 0 else '#B71C1C')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    else:
        cell.set_facecolor('#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title('Logistic Regression — Baseline vs Tuned Results Table',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

#### Final Results Summary

In [ ]:
best_p = search_q3.best_params_

summary_tuned = [
    ['Model',             'Logistic Regression (Tuned — GridSearchCV)'],
    ['Accuracy',          f'{acc_tuned:.4f}'],
    ['Macro Precision',   f'{macro_p_t:.4f}'],
    ['Macro Recall',      f'{macro_r_t:.4f}'],
    ['Macro F1-Score',    f'{macro_f1_t:.4f}'],
    ['Weighted F1-Score', f'{weighted_f1_t:.4f}'],
    ['class_weight',      'balanced'],
    ['max_features',      str(best_p['tfidf__max_features'])],
    ['ngram_range',       '(1, 2)'],
    ['sublinear_tf',      'True'],
    ['min_df',            '1'],
    ['C',                 str(best_p['lr__C'])],
    ['penalty',           'l2'],
    ['solver',            'lbfgs'],
    ['Search method',     'GridSearchCV (exhaustive)'],
    ['Combinations',      f'{n_combos}  (2 × 4)'],
    ['CV folds',          '3'],
    ['Total fits',        str(n_combos * 3)],
    ['Best CV Macro F1',  f'{search_q3.best_score_:.4f}'],
]

fig, ax = plt.subplots(figsize=(11, 7))
ax.axis('off')
fig.patch.set_facecolor('white')
tbl_fin = ax.table(
    cellText=summary_tuned,
    colLabels=['Property', 'Value'],
    cellLoc='left', loc='center',
    colWidths=[0.40, 0.60]
)
tbl_fin.auto_set_font_size(False); tbl_fin.set_fontsize(11); tbl_fin.scale(1, 1.65)
for (r, c), cell in tbl_fin.get_celld().items():
    if r == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    else:
        cell.set_facecolor('#FFFFFF')
    cell.set_edgecolor('#BBDEFB')

plt.title('Logistic Regression — Tuned Results Summary (GridSearchCV)',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print('=' * 55)
print(' TUNING COMPLETE (GridSearchCV)')
print('=' * 55)
print(f'Baseline Macro F1  : {macro_f1:.4f}')
print(f'Tuned Macro F1     : {macro_f1_t:.4f}  ({macro_f1_t - macro_f1:+.4f})')
print(f'Baseline Accuracy  : {accuracy:.4f}')
print(f'Tuned Accuracy     : {acc_tuned:.4f}  ({acc_tuned - accuracy:+.4f})')
print(f'Neutral F1 (base)  : {f1_base_per[2]:.4f}')
print(f'Neutral F1 (tuned) : {f1_tuned_per[2]:.4f}  ({f1_tuned_per[2]-f1_base_per[2]:+.4f})')
print('=' * 55)